# LeetCode 847 — Shortest Path Visiting All Nodes bằng Python

- Link: https://leetcode.com/problems/shortest-path-visiting-all-nodes/
- Độ khó: **Hard**
- Chủ đề: BFS đa nguồn trên trạng thái `(mask, node)`, Bitmask, Quy hoạch động Held–Karp, Khoảng cách mọi cặp

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def shortestPathLength(self, graph: List[List[int]]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho một đồ thị vô hướng, liên thông, `n <= 12` đỉnh. Tìm độ dài (số cạnh) của đường đi **ngắn nhất thăm mọi đỉnh**. Được bắt đầu và kết thúc ở bất kỳ đỉnh nào, được **đi lại** đỉnh và cạnh nhiều lần.

Đây là "người du lịch" (TSP) phiên bản đồ thị không trọng số, không cần quay về. TSP là bài NP-khó, nhưng `n <= 12` gợi ý ngay: **bitmask** (tập đỉnh đã thăm) có `2¹² = 4096` giá trị. Trạng thái `(tập đã thăm, đỉnh hiện tại)` chỉ có `4096 × 12 ≈ 5·10⁴` → BFS hoặc DP trên trạng thái này là đủ nhanh.

Sau notebook này bạn sẽ:

1. Biểu diễn tập con bằng **bitmask** và thao tác bit cơ bản (`|`, `&`, `<<`).
2. Hiểu vì sao trạng thái phải gồm **cả** tập đã thăm **và** đỉnh hiện tại.
3. Cài **BFS đa nguồn** trên trạng thái `(mask, node)`.
4. Cài **DP Held–Karp** `dp[mask][v]` dùng khoảng cách mọi cặp (BFS từ mỗi đỉnh).
5. Đối chiếu với brute force duyệt mọi hoán vị.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Đồ thị vô hướng liên thông gồm `n` đỉnh `0..n−1`, cho dưới dạng danh sách kề `graph` (`graph[i]` là các đỉnh kề với `i`). Trả về số cạnh ít nhất của một đường đi (có thể lặp đỉnh, lặp cạnh) đi qua **mọi** đỉnh ít nhất một lần. Điểm đầu và điểm cuối tuỳ ý.

**Ví dụ 1**

```text
Input: graph = [[1,2,3],[0],[0],[0]]
Output: 4
Giải thích: một đường đi: [1,0,2,0,3].
```

```text
    1
    |
2 — 0 — 3
```

**Ví dụ 2**

```text
Input: graph = [[1],[0,2,4],[1,3,4],[2],[1,2]]
Output: 4
Giải thích: một đường đi: [0,1,4,2,3].
```

**Ràng buộc**

- `n == graph.length`, `1 <= n <= 12`
- `0 <= graph[i].length < n`
- `graph[i]` không chứa `i`; nếu `graph[a]` chứa `b` thì `graph[b]` chứa `a`.
- Đồ thị liên thông.

In [1]:
from typing import List

EXAMPLES = [
    ([[1, 2, 3], [0], [0], [0]], 4),
    ([[1], [0, 2, 4], [1, 3, 4], [2], [1, 2]], 4),
    ([[]], 0),                                   # một đỉnh
    ([[1], [0]], 1),
    ([[1], [0, 2], [1, 3], [2]], 3),             # đường thẳng 4 đỉnh
]
for g, exp in EXAMPLES:
    print(g, "-> kỳ vọng", exp)

[[1, 2, 3], [0], [0], [0]] -> kỳ vọng 4
[[1], [0, 2, 4], [1, 3, 4], [2], [1, 2]] -> kỳ vọng 4
[[]] -> kỳ vọng 0
[[1], [0]] -> kỳ vọng 1
[[1], [0, 2], [1, 3], [2]] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Bitmask biểu diễn tập con

Tập con của `{0..n−1}` ↔ số nguyên `mask` có bit `i` bằng 1 nếu `i` thuộc tập.

- Thêm `i`: `mask | (1 << i)`
- Kiểm tra `i`: `mask >> i & 1` hoặc `mask & (1 << i)`
- Tập đầy đủ: `(1 << n) − 1`

In [2]:
mask = 0
mask |= 1 << 0
mask |= 1 << 3
print(bin(mask), mask >> 3 & 1, mask >> 2 & 1, bin((1 << 4) - 1))

0b1001 1 0 0b1111


### 3.2. Đếm bit và duyệt các bit bật

`bin(mask).count("1")` (hoặc `mask.bit_count()` từ Python 3.10) đếm số phần tử.

In [3]:
mask = 0b1011
print(bin(mask).count("1"), [i for i in range(4) if mask >> i & 1])

3 [0, 1, 3]


### 3.3. BFS đa nguồn

Đưa **nhiều** trạng thái xuất phát vào hàng đợi cùng lúc với khoảng cách 0. BFS khi đó tính khoảng cách ngắn nhất từ **tập** nguồn.

In [4]:
from collections import deque

def multi_source_bfs(adj, sources):
    dist = {s: 0 for s in sources}
    q = deque(sources)
    while q:
        u = q.popleft()
        for v in adj[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

print(multi_source_bfs([[1], [0, 2], [1, 3], [2]], [0, 3]))

{0: 0, 3: 0, 1: 1, 2: 1}


### 3.4. Khoảng cách mọi cặp bằng BFS từ mỗi đỉnh

Đồ thị không trọng số: `n` lần BFS, mỗi lần `O(n + m)`.

In [5]:
def all_pairs(graph):
    n = len(graph)
    D = []
    for s in range(n):
        d = multi_source_bfs(graph, [s])
        D.append([d[v] for v in range(n)])
    return D

for row in all_pairs([[1, 2, 3], [0], [0], [0]]):
    print(row)

[0, 1, 1, 1]
[1, 0, 2, 2]
[1, 2, 0, 2]
[1, 2, 2, 0]


### 3.5. Hoán vị với `itertools.permutations`

In [6]:
from itertools import permutations
print(list(permutations([0, 1, 2])))

[(0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0)]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Vì sao cần đi lại?

Ví dụ 1 là hình sao: tâm 0 nối 1, 2, 3. Không có đường đơn nào thăm cả 4 đỉnh (lá nào cũng chỉ có một cạnh). Phải quay lại tâm: `1 → 0 → 2 → 0 → 3` = 4 cạnh.

### 4.2. Trạng thái phải là `(mask, node)`

Hai lần đứng ở đỉnh 0 trong ví dụ trên **khác nhau**: lần đầu đã thăm `{0, 1}`, lần sau đã thăm `{0, 1, 2}`. Chỉ ghi nhớ "đã thăm đỉnh 0" thì BFS sẽ không cho quay lại 0 → sai. Trạng thái đúng là **(tập đã thăm, đỉnh hiện tại)**.

### 4.3. Tính tay BFS ví dụ 1 (một phần)

Nguồn: `({0},0), ({1},1), ({2},2), ({3},3)` với khoảng cách 0.

| bước | một số trạng thái mới |
|---|---|
| 1 | `({0,1},1), ({0,2},2), ({0,3},3), ({0,1},0), ({0,2},0), ({0,3},0)` |
| 2 | `({0,1,2},2)` từ `({0,1},0)`, ... |
| 3 | `({0,1,2},0)` ... |
| 4 | `({0,1,2,3},3)` — mask đầy đủ → đáp án **4** |

In [7]:
def show_mask(mask, n):
    return "{" + ",".join(str(i) for i in range(n) if mask >> i & 1) + "}"

print(show_mask(0b0111, 4), show_mask(0b1111, 4))

{0,1,2} {0,1,2,3}


## 5. Cách 1 — Brute Force: thử mọi thứ tự thăm

**Ý tưởng.** Đường đi tối ưu thăm các đỉnh **lần đầu** theo một thứ tự `p₀, p₁, ..., p_{n−1}`. Giữa hai đỉnh liên tiếp trong thứ tự, đi theo đường ngắn nhất. Vậy đáp án = `min` trên mọi hoán vị của `Σ D[pᵢ][pᵢ₊₁]`.

In [8]:
def shortest_brute(graph):
    n = len(graph)
    D = all_pairs(graph)
    best = float('inf')
    for p in permutations(range(n)):
        total = sum(D[p[i]][p[i + 1]] for i in range(n - 1))
        best = min(best, total)
    return best

for g, exp in EXAMPLES:
    assert shortest_brute(g) == exp
print("Brute force: OK")

Brute force: OK


**Độ phức tạp.** `n!` hoán vị × `O(n)` → `12! · 12 ≈ 5.7·10⁹` — quá chậm (chỉ chạy được tới `n ≈ 8–9`).

**Nút thắt.** Nhiều hoán vị có cùng **tiền tố dạng tập**: thứ tự `0,1,2,3,...` và `1,0,2,3,...` đều tới đỉnh 2 sau khi thăm `{0,1,2}`. Phần còn lại của hành trình chỉ phụ thuộc vào **tập đã thăm** và **đỉnh hiện tại**, không phụ thuộc thứ tự trước đó → gộp lại: `n! → 2ⁿ · n`.

## 6. Bước đột phá

**Insight 1 — Trạng thái `(mask, v)`.** Tương lai của hành trình chỉ phụ thuộc vào tập đỉnh đã thăm và đỉnh đang đứng. Số trạng thái `2ⁿ · n ≤ 49 152`.

**Insight 2 — BFS vì mọi cạnh có trọng số 1.** Từ `(mask, v)` đi sang `(mask | 1<<u, u)` với mọi `u` kề `v`, chi phí 1. BFS từ **mọi** nguồn `(1<<v, v)` (vì được bắt đầu ở bất kỳ đâu). Lần đầu lấy ra một trạng thái có `mask` đầy đủ → đáp án.

**Insight 3 — DP Held–Karp.** `dp[mask][v]` = độ dài ngắn nhất của hành trình **bắt đầu ở đâu đó**, đã thăm đúng tập `mask` (theo nghĩa các đỉnh thăm lần đầu), và **kết thúc ở `v`**. Chuyển: `dp[mask | 1<<u][u] = min(dp[mask][v] + D[v][u])` với `u ∉ mask`. Khởi tạo `dp[1<<v][v] = 0`. Đáp án `min_v dp[full][v]`.

**Quy tắc nếu…thì:**
- *Nếu* `n == 1` *thì* đáp án 0.
- *Nếu* trạng thái `(mask, v)` đã thăm *thì* bỏ qua (BFS cho khoảng cách nhỏ nhất ngay lần đầu).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — BFS đa nguồn trên `(mask, node)`

In [9]:
def shortest_bfs(graph):
    n = len(graph)
    full = (1 << n) - 1
    if n == 1:
        return 0
    seen = [[False] * n for _ in range(1 << n)]
    q = deque()
    for v in range(n):
        seen[1 << v][v] = True
        q.append((1 << v, v, 0))
    while q:
        mask, v, d = q.popleft()
        for u in graph[v]:
            nm = mask | (1 << u)
            if nm == full:
                return d + 1
            if not seen[nm][u]:
                seen[nm][u] = True
                q.append((nm, u, d + 1))
    return -1

for g, exp in EXAMPLES:
    assert shortest_bfs(g) == exp
print("BFS (mask, node): OK")

BFS (mask, node): OK


Giải thích:

- `seen[mask][v]`: mảng 2 chiều `2ⁿ × n` (nhanh hơn `set` các tuple).
- Kiểm tra `nm == full` **khi sinh** trạng thái con → trả `d + 1` sớm, tiết kiệm một lớp BFS.
- Trường hợp `n == 1` xử lý riêng vì không có cạnh để sinh trạng thái con.

Code nộp LeetCode cho cách 2:

In [10]:
class SolutionBFS:
    def shortestPathLength(self, graph: List[List[int]]) -> int:
        n = len(graph)
        if n == 1:
            return 0
        full = (1 << n) - 1
        seen = [[False] * n for _ in range(1 << n)]
        q = deque()
        for v in range(n):
            seen[1 << v][v] = True
            q.append((1 << v, v))
        steps = 0
        while q:
            steps += 1
            for _ in range(len(q)):
                mask, v = q.popleft()
                for u in graph[v]:
                    nm = mask | (1 << u)
                    if nm == full:
                        return steps
                    if not seen[nm][u]:
                        seen[nm][u] = True
                        q.append((nm, u))
        return -1

for g, exp in EXAMPLES:
    assert SolutionBFS().shortestPathLength(g) == exp
print("SolutionBFS: OK")

SolutionBFS: OK


**Độ phức tạp cách 2:** mỗi trạng thái `(mask, v)` xử lý một lần, duyệt `deg(v)` cạnh → `O(2ⁿ · Σ deg) = O(2ⁿ · m) ⊆ O(2ⁿ · n²)`. Bộ nhớ `O(2ⁿ · n)`.

### 7.2. Cách 3 — DP Held–Karp với khoảng cách mọi cặp

Duyệt `mask` tăng dần (mọi mask con có giá trị nhỏ hơn mask cha khi thêm bit → thứ tự tăng dần là thứ tự topo hợp lệ).

In [11]:
def shortest_held_karp(graph):
    n = len(graph)
    D = all_pairs(graph)
    INF = float('inf')
    full = (1 << n) - 1
    dp = [[INF] * n for _ in range(1 << n)]
    for v in range(n):
        dp[1 << v][v] = 0
    for mask in range(1, 1 << n):
        for v in range(n):
            cur = dp[mask][v]
            if cur == INF:
                continue
            for u in range(n):
                if mask >> u & 1:
                    continue
                nm = mask | (1 << u)
                if cur + D[v][u] < dp[nm][u]:
                    dp[nm][u] = cur + D[v][u]
    return min(dp[full])

for g, exp in EXAMPLES:
    assert shortest_held_karp(g) == exp
print("Held–Karp: OK")

Held–Karp: OK


Ở đây `mask` là tập các đỉnh **được "chốt" theo thứ tự thăm lần đầu**; đường ngắn nhất `D[v][u]` có thể đi qua đỉnh đã/ chưa thăm — không sao: đi qua đỉnh chưa thăm mà không "chốt" nó chỉ có thể làm đường dài hơn hoặc bằng, và DP vẫn tìm được thứ tự tối ưu (xem mục 10).

Code nộp LeetCode cho cách 3:

In [12]:
class SolutionHeldKarp:
    def shortestPathLength(self, graph: List[List[int]]) -> int:
        n = len(graph)
        D = [[0] * n for _ in range(n)]
        for s in range(n):
            dist = [-1] * n
            dist[s] = 0
            q = deque([s])
            while q:
                x = q.popleft()
                for y in graph[x]:
                    if dist[y] < 0:
                        dist[y] = dist[x] + 1
                        q.append(y)
            D[s] = dist
        INF = 10 ** 9
        full = (1 << n) - 1
        dp = [[INF] * n for _ in range(1 << n)]
        for v in range(n):
            dp[1 << v][v] = 0
        for mask in range(1, 1 << n):
            row = dp[mask]
            for v in range(n):
                cur = row[v]
                if cur >= INF:
                    continue
                Dv = D[v]
                for u in range(n):
                    if not mask >> u & 1:
                        nm = mask | (1 << u)
                        if cur + Dv[u] < dp[nm][u]:
                            dp[nm][u] = cur + Dv[u]
        return min(dp[full])

for g, exp in EXAMPLES:
    assert SolutionHeldKarp().shortestPathLength(g) == exp
print("SolutionHeldKarp: OK")

SolutionHeldKarp: OK


**Độ phức tạp cách 3:** `O(n·(n + m))` cho khoảng cách + `O(2ⁿ · n²)` cho DP; bộ nhớ `O(2ⁿ · n)`. Với `n = 12`: `4096 · 144 ≈ 6·10⁵` phép — nhanh.

### 7.3. Cách 4 — DP "cạnh kề" lặp tới khi ổn định (Bellman–Ford trên mask)

Không cần khoảng cách mọi cặp: `dp[mask][v]` = đường ngắn nhất kết thúc ở `v` với tập đã thăm **đúng** `mask` (theo nghĩa đi từng cạnh). Chuyển theo cạnh `v → u`: `dp[mask | 1<<u][u] ≤ dp[mask][v] + 1`. Vì `u` có thể đã nằm trong `mask` (quay lại), chuyển có thể giữ nguyên `mask` → trong một `mask` cố định phải **lặp** cho tới khi không cải thiện (giống Bellman–Ford), hoặc chạy BFS nội bộ trong mỗi `mask`.

In [13]:
def shortest_mask_relax(graph):
    n = len(graph)
    INF = float('inf')
    full = (1 << n) - 1
    dp = [[INF] * n for _ in range(1 << n)]
    for v in range(n):
        dp[1 << v][v] = 0
    for mask in range(1, 1 << n):
        # BFS nội bộ trong cùng mask: lan dp qua các cạnh giữ nguyên mask (đi lại đỉnh đã thăm)
        q = deque(v for v in range(n) if dp[mask][v] < INF)
        while q:
            v = q.popleft()
            for u in graph[v]:
                if mask >> u & 1 and dp[mask][v] + 1 < dp[mask][u]:
                    dp[mask][u] = dp[mask][v] + 1
                    q.append(u)
        # chuyển sang mask lớn hơn
        for v in range(n):
            if dp[mask][v] == INF:
                continue
            for u in graph[v]:
                if not mask >> u & 1:
                    nm = mask | (1 << u)
                    dp[nm][u] = min(dp[nm][u], dp[mask][v] + 1)
    return min(dp[full])

for g, exp in EXAMPLES:
    assert shortest_mask_relax(g) == exp
print("DP theo cạnh + lan trong mask: OK")

DP theo cạnh + lan trong mask: OK


Lưu ý: BFS nội bộ dùng hàng đợi với "giảm khoá" kiểu SPFA — đúng vì mọi cạnh trọng số 1 nhưng giá trị đầu khác nhau; mỗi lần cải thiện thì đẩy lại. Cách này minh hoạ rằng trạng thái `(mask, v)` có thể được tính theo thứ tự `mask` tăng dần, bên trong một `mask` thì là bài đường đi ngắn nhất.

Code nộp LeetCode cho cách 4:

In [14]:
class SolutionMaskRelax:
    def shortestPathLength(self, graph: List[List[int]]) -> int:
        n = len(graph)
        INF = 10 ** 9
        full = (1 << n) - 1
        dp = [[INF] * n for _ in range(1 << n)]
        for v in range(n):
            dp[1 << v][v] = 0
        for mask in range(1, 1 << n):
            row = dp[mask]
            q = deque(v for v in range(n) if row[v] < INF)
            while q:
                v = q.popleft()
                for u in graph[v]:
                    if mask >> u & 1 and row[v] + 1 < row[u]:
                        row[u] = row[v] + 1
                        q.append(u)
            for v in range(n):
                if row[v] < INF:
                    for u in graph[v]:
                        if not mask >> u & 1:
                            nm = mask | (1 << u)
                            if row[v] + 1 < dp[nm][u]:
                                dp[nm][u] = row[v] + 1
        return min(dp[full])

for g, exp in EXAMPLES:
    assert SolutionMaskRelax().shortestPathLength(g) == exp
print("SolutionMaskRelax: OK")

SolutionMaskRelax: OK


**Độ phức tạp cách 4:** `O(2ⁿ · n · m)` xấu nhất (SPFA trong mỗi mask), thực tế nhanh. Bộ nhớ `O(2ⁿ · n)`.

## 8. TRACE từng bước

Trace cách 2 (BFS) trên ví dụ 1, in từng lớp BFS với trạng thái dạng tập.

In [15]:
def trace_bfs(graph, max_print=8):
    n = len(graph)
    full = (1 << n) - 1
    if n == 1:
        print("n = 1 -> 0"); return 0
    seen = {(1 << v, v) for v in range(n)}
    layer = [(1 << v, v) for v in range(n)]
    print("lớp 0:", [(show_mask(m, n), v) for m, v in layer])
    steps = 0
    while layer:
        steps += 1
        nxt = []
        for mask, v in layer:
            for u in graph[v]:
                nm = mask | (1 << u)
                if nm == full:
                    print(f"lớp {steps}: từ ({show_mask(mask, n)}, {v}) đi tới {u} -> mask đầy đủ. ĐÁP ÁN = {steps}")
                    return steps
                if (nm, u) not in seen:
                    seen.add((nm, u))
                    nxt.append((nm, u))
        shown = [(show_mask(m, n), v) for m, v in nxt[:max_print]]
        print(f"lớp {steps}: {len(nxt)} trạng thái mới, ví dụ {shown}{' ...' if len(nxt) > max_print else ''}")
        layer = nxt
    return -1

trace_bfs([[1, 2, 3], [0], [0], [0]])

lớp 0: [('{0}', 0), ('{1}', 1), ('{2}', 2), ('{3}', 3)]
lớp 1: 6 trạng thái mới, ví dụ [('{0,1}', 1), ('{0,2}', 2), ('{0,3}', 3), ('{0,1}', 0), ('{0,2}', 0), ('{0,3}', 0)]
lớp 2: 6 trạng thái mới, ví dụ [('{0,1,2}', 2), ('{0,1,3}', 3), ('{0,1,2}', 1), ('{0,2,3}', 3), ('{0,1,3}', 1), ('{0,2,3}', 2)]
lớp 3: 3 trạng thái mới, ví dụ [('{0,1,2}', 0), ('{0,1,3}', 0), ('{0,2,3}', 0)]
lớp 4: từ ({0,1,2}, 0) đi tới 3 -> mask đầy đủ. ĐÁP ÁN = 4


4

Bảng trạng thái (một nhánh dẫn tới đáp án):

| bước | trạng thái (mask, v) | ghi chú |
|---|---|---|
| 0 | ({1}, 1) | nguồn |
| 1 | ({0,1}, 0) | đi 1 → 0 |
| 2 | ({0,1,2}, 2) | đi 0 → 2 |
| 3 | ({0,1,2}, 0) | quay lại 0 (mask không đổi, đỉnh đổi) |
| 4 | ({0,1,2,3}, 3) | đầy đủ ✔ |

Trace cách 3 (Held–Karp) — in các giá trị `dp[mask][v]` hữu hạn theo mask cho ví dụ 2:

In [16]:
def trace_held_karp(graph):
    n = len(graph)
    D = all_pairs(graph)
    INF = float('inf')
    dp = [[INF] * n for _ in range(1 << n)]
    for v in range(n):
        dp[1 << v][v] = 0
    for mask in range(1, 1 << n):
        for v in range(n):
            if dp[mask][v] == INF:
                continue
            for u in range(n):
                if not mask >> u & 1:
                    nm = mask | (1 << u)
                    dp[nm][u] = min(dp[nm][u], dp[mask][v] + D[v][u])
    for size in (2, n):
        print(f"--- các mask có {size} phần tử ---")
        for mask in range(1 << n):
            if bin(mask).count("1") == size:
                vals = {v: dp[mask][v] for v in range(n) if dp[mask][v] < INF}
                if size == n or mask in (0b00011, 0b00110, 0b10010):
                    print(show_mask(mask, n), vals)
    return min(dp[(1 << n) - 1])

print("đáp án:", trace_held_karp([[1], [0, 2, 4], [1, 3, 4], [2], [1, 2]]))

--- các mask có 2 phần tử ---
{0,1} {0: 1, 1: 1}
{1,2} {1: 1, 2: 1}
{1,4} {1: 1, 4: 1}
--- các mask có 5 phần tử ---
{0,1,2,3,4} {0: 4, 1: 5, 2: 5, 3: 4, 4: 5}
đáp án: 4


## 9. Code nộp LeetCode

Chọn cách 2 (BFS đa nguồn trên `(mask, node)`): trực quan, `O(2ⁿ · m)`.

In [17]:
from collections import deque
from typing import List

class Solution:
    def shortestPathLength(self, graph: List[List[int]]) -> int:
        n = len(graph)
        if n == 1:
            return 0
        full = (1 << n) - 1
        seen = [[False] * n for _ in range(1 << n)]
        q = deque()
        for v in range(n):                      # được bắt đầu ở bất kỳ đỉnh nào
            seen[1 << v][v] = True
            q.append((1 << v, v))
        steps = 0
        while q:
            steps += 1
            for _ in range(len(q)):             # xử lý trọn một lớp
                mask, v = q.popleft()
                for u in graph[v]:
                    nm = mask | (1 << u)
                    if nm == full:              # đã thăm đủ mọi đỉnh
                        return steps
                    if not seen[nm][u]:
                        seen[nm][u] = True
                        q.append((nm, u))
        return -1

for g, exp in EXAMPLES:
    assert Solution().shortestPathLength(g) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

**Mô hình BFS.** Xây đồ thị trạng thái `H`: đỉnh `(mask, v)` với `v ∈ mask`; cạnh `(mask, v) → (mask ∪ {u}, u)` cho mọi `u` kề `v`. Một đường đi `w₀, w₁, ..., w_L` trong đồ thị gốc tương ứng một-một với đường đi trong `H` bắt đầu ở `({w₀}, w₀)`, trong đó mask tại bước `i` là `{w₀, ..., wᵢ}`. Đường đi gốc thăm mọi đỉnh ⇔ đường đi trong `H` kết thúc ở trạng thái có mask đầy đủ. Vậy đáp án = khoảng cách ngắn nhất trong `H` từ tập nguồn `{({v}, v)}` tới tập đích `{(full, ·)}`. BFS đa nguồn tính đúng khoảng cách này (mọi cạnh trọng số 1).

**Held–Karp.** Xét hành trình tối ưu; liệt kê các đỉnh theo thứ tự **thăm lần đầu** `p₀, ..., p_{n−1}`. Đoạn giữa `pᵢ` và `pᵢ₊₁` có độ dài ≥ `D[pᵢ][pᵢ₊₁]`, nên độ dài ≥ `Σ D[pᵢ][pᵢ₊₁]`. Ngược lại, ghép các đường ngắn nhất `pᵢ → pᵢ₊₁` cho một hành trình hợp lệ có đúng độ dài đó. Vậy đáp án = `min` theo hoán vị `Σ D[pᵢ][pᵢ₊₁]` (đó cũng là lý do brute force đúng). DP `dp[mask][v]` = `min` của tổng trên các tiền tố hoán vị có tập phần tử `mask` và kết thúc ở `v` — thuộc tính cấu trúc con tối ưu chuẩn của TSP: phần tiếp theo chỉ phụ thuộc `(mask, v)`.

**Cách 4.** Giống BFS nhưng tính theo thứ tự `mask` tăng: mọi cạnh trong `H` hoặc giữ nguyên mask (quay lại đỉnh đã thăm) hoặc tăng mask. Với mask cố định, giải đường ngắn nhất nội bộ (SPFA) rồi đẩy sang mask lớn hơn — đúng vì không có cạnh nào làm giảm mask.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Hoán vị | `O(n! · n)` | `O(n²)` |
| 2. BFS `(mask, node)` | `O(2ⁿ · m)` | `O(2ⁿ · n)` |
| 3. Held–Karp | `O(2ⁿ · n²)` | `O(2ⁿ · n)` |
| 4. DP theo cạnh + SPFA | `O(2ⁿ · n · m)` xấu nhất | `O(2ⁿ · n)` |

In [18]:
import random, time

def random_connected_graph(n, extra_p, rnd):
    edges = set()
    for i in range(1, n):
        j = rnd.randrange(i)
        edges.add((i, j))
    for i in range(n):
        for j in range(i + 1, n):
            if rnd.random() < extra_p:
                edges.add((j, i))
    g = [[] for _ in range(n)]
    for a, b in edges:
        if b not in g[a]:
            g[a].append(b); g[b].append(a)
    return g

rnd = random.Random(847)
g_dense = [[j for j in range(12) if j != i] for i in range(12)]    # đồ thị đầy đủ
g_sparse = random_connected_graph(12, 0.1, rnd)
for name, fn in [("bfs", shortest_bfs), ("held_karp", shortest_held_karp), ("mask_relax", shortest_mask_relax),
                 ("Solution", lambda g: Solution().shortestPathLength(g))]:
    t0 = time.perf_counter()
    res = (fn(g_dense), fn(g_sparse))
    print(f"{name:10}: {res}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")

bfs       : (11, 11)  (29 ms)


held_karp : (11, 11)  (56 ms)


mask_relax: (11, 11)  (80 ms)


Solution  : (11, 11)  (29 ms)


## 12. Bộ test

In [19]:
methods = {
    "brute": shortest_brute,
    "bfs": shortest_bfs,
    "held_karp": shortest_held_karp,
    "mask_relax": shortest_mask_relax,
    "SolutionBFS": lambda g: SolutionBFS().shortestPathLength(g),
    "SolutionHeldKarp": lambda g: SolutionHeldKarp().shortestPathLength(g),
    "SolutionMaskRelax": lambda g: SolutionMaskRelax().shortestPathLength(g),
    "Solution": lambda g: Solution().shortestPathLength(g),
}

# 12.1 Ví dụ + edge cases
star6 = [[1, 2, 3, 4, 5], [0], [0], [0], [0], [0]]
cycle5 = [[1, 4], [0, 2], [1, 3], [2, 4], [3, 0]]
complete4 = [[1, 2, 3], [0, 2, 3], [0, 1, 3], [0, 1, 2]]
edge = EXAMPLES + [(star6, 8), (cycle5, 4), (complete4, 3)]
for name, fn in methods.items():
    for g, exp in edge:
        assert fn(g) == exp, (name, g, fn(g), exp)
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed: đối chiếu brute force (n <= 7)
for seed in range(300):
    r_ = random.Random(seed)
    n = r_.randint(1, 7)
    g = random_connected_graph(n, r_.random() * 0.5, r_)
    ref = shortest_brute(g)
    for name, fn in methods.items():
        assert fn(g) == ref, (name, g)
print("300 đồ thị ngẫu nhiên: khớp brute force")

# 12.3 n = 10..12: các cách nhanh khớp nhau
for seed in range(6):
    r_ = random.Random(100 + seed)
    g = random_connected_graph(r_.randint(10, 12), 0.15, r_)
    vals = {name: fn(g) for name, fn in methods.items() if name != "brute"}
    assert len(set(vals.values())) == 1, vals
print("n = 10..12: các cách nhanh khớp nhau")

Ví dụ + edge cases: OK


300 đồ thị ngẫu nhiên: khớp brute force


n = 10..12: các cách nhanh khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — `visited` chỉ theo đỉnh (không cho quay lại)

**Code sai:**

In [20]:
def wrong_visited_node(graph):
    n = len(graph)
    full = (1 << n) - 1
    if n == 1:
        return 0
    seen = set(range(n))                               # SAI: chỉ nhớ đỉnh
    q = deque((1 << v, v, 0) for v in range(n))
    best = -1
    while q:
        mask, v, d = q.popleft()
        for u in graph[v]:
            nm = mask | (1 << u)
            if nm == full:
                return d + 1
            if (u, nm) not in seen and u not in seen:
                seen.add(u)
                q.append((nm, u, d + 1))
    return best

print("sai:", wrong_visited_node([[1, 2, 3], [0], [0], [0]]), " đúng:", 4)

sai: -1  đúng: 4


**Vì sao sai:** hình sao bắt buộc quay lại tâm; chặn thăm lại đỉnh thì không bao giờ phủ hết → trả `-1`. **Code đúng:** `visited` theo cặp `(mask, v)`.

### Lỗi 2 — Chỉ bắt đầu BFS từ đỉnh 0

**Code sai:**

In [21]:
def wrong_single_source(graph):
    n = len(graph)
    full = (1 << n) - 1
    if n == 1:
        return 0
    seen = {(1, 0)}
    q = deque([(1, 0, 0)])                             # SAI: chỉ xuất phát từ 0
    while q:
        mask, v, d = q.popleft()
        for u in graph[v]:
            nm = mask | (1 << u)
            if nm == full:
                return d + 1
            if (nm, u) not in seen:
                seen.add((nm, u)); q.append((nm, u, d + 1))
    return -1

print("sai:", wrong_single_source([[1, 2, 3], [0], [0], [0]]), " đúng:", 4)

sai: 5  đúng: 4


**Vì sao sai:** từ tâm 0 phải đi `0→1→0→2→0→3` = 5 cạnh; bắt đầu từ lá 1 chỉ cần 4. Đề cho phép bắt đầu ở bất kỳ đâu → **đa nguồn**.

### Lỗi 3 — Held–Karp dùng `graph` (kề trực tiếp) thay vì khoảng cách ngắn nhất

**Code sai:**

In [22]:
def wrong_hk_adjacent(graph):
    n = len(graph)
    INF = float('inf')
    dp = [[INF] * n for _ in range(1 << n)]
    for v in range(n):
        dp[1 << v][v] = 0
    for mask in range(1, 1 << n):
        for v in range(n):
            if dp[mask][v] == INF:
                continue
            for u in graph[v]:                         # SAI: chỉ đi tới đỉnh kề CHƯA thăm
                if not mask >> u & 1:
                    nm = mask | (1 << u)
                    dp[nm][u] = min(dp[nm][u], dp[mask][v] + 1)
    return min(dp[(1 << n) - 1])

print("sai:", wrong_hk_adjacent([[1, 2, 3], [0], [0], [0]]), " đúng:", 4)

sai: inf  đúng: 4


**Vì sao sai:** đây là tìm **đường đi Hamilton** (không lặp đỉnh) — hình sao không có → `inf`. **Code đúng:** dùng `D[v][u]` (khoảng cách ngắn nhất, cho phép đi qua đỉnh đã thăm), hoặc cho phép chuyển giữ nguyên mask như cách 4.

### Lỗi 4 — Quên trường hợp `n == 1`

BFS kiểm tra `nm == full` chỉ khi sinh con; với một đỉnh không có cạnh nào → vòng lặp kết thúc, trả `-1`. Phải `return 0` khi `n == 1` (hoặc kiểm tra mask đầy đủ khi khởi tạo).

In [23]:
def wrong_n1(graph):
    n = len(graph); full = (1 << n) - 1
    q = deque((1 << v, v, 0) for v in range(n)); seen = {(1 << v, v) for v in range(n)}
    while q:
        mask, v, d = q.popleft()
        for u in graph[v]:
            nm = mask | (1 << u)
            if nm == full:
                return d + 1
            if (nm, u) not in seen:
                seen.add((nm, u)); q.append((nm, u, d + 1))
    return -1

print("sai:", wrong_n1([[]]), " đúng:", 0)

sai: -1  đúng: 0


## 14. Mẹo & nhận dạng pattern

- **`n <= 12..20` và "thăm mọi đỉnh / chọn tập con"** → bitmask; trạng thái thường là `(mask, vị trí cuối)`.
- **Trọng số 1** → BFS trên trạng thái; **trọng số tuỳ ý** → Held–Karp/Dijkstra.
- **Được bắt đầu ở đâu cũng được** → BFS **đa nguồn**.
- **Cho phép đi lại** → dùng khoảng cách ngắn nhất `D[u][v]` giữa các đỉnh "quan trọng" rồi TSP trên chúng.
- Bài liên quan: **943 Find the Shortest Superstring**, **864 Shortest Path to Get All Keys**, **1494 Parallel Courses II**, **1125 Smallest Sufficient Team**, **980 Unique Paths III**.

## 15. Biến thể

1. **Phải quay về điểm xuất phát** (chu trình): Held–Karp cố định đỉnh đầu 0, đáp án `min_v dp[full][v] + D[v][0]`.
2. **Cạnh có trọng số**: Held–Karp với `D` từ Dijkstra/Floyd; hoặc Dijkstra trên `(mask, v)`.
3. **Chỉ cần thăm một tập đỉnh quan trọng `K`** (`|K|` nhỏ, `n` lớn): BFS từ mỗi đỉnh trong `K`, TSP trên `K` (như 864 với chìa khoá).
4. **In ra hành trình**: lưu `parent` cho trạng thái BFS rồi truy ngược.
5. **Điểm xuất phát cố định**: BFS từ một nguồn `({s}, s)`.

Minh hoạ biến thể 1 và 4:

In [24]:
def shortest_cycle_all(graph):
    n = len(graph)
    if n == 1:
        return 0
    D = all_pairs(graph)
    INF = float('inf')
    dp = [[INF] * n for _ in range(1 << n)]
    dp[1][0] = 0
    for mask in range(1, 1 << n, 2):                 # chỉ mask chứa đỉnh 0
        for v in range(n):
            if dp[mask][v] == INF:
                continue
            for u in range(n):
                if not mask >> u & 1:
                    nm = mask | (1 << u)
                    dp[nm][u] = min(dp[nm][u], dp[mask][v] + D[v][u])
    full = (1 << n) - 1
    return min(dp[full][v] + D[v][0] for v in range(n))

def path_all_nodes(graph):
    n = len(graph)
    if n == 1:
        return [0]
    full = (1 << n) - 1
    parent = {(1 << v, v): None for v in range(n)}
    q = deque(parent.keys())
    while q:
        st = q.popleft()
        mask, v = st
        for u in graph[v]:
            ns = (mask | (1 << u), u)
            if ns not in parent:
                parent[ns] = st
                if ns[0] == full:
                    path, cur = [], ns
                    while cur:
                        path.append(cur[1]); cur = parent[cur]
                    return path[::-1]
                q.append(ns)

print("chu trình hình sao:", shortest_cycle_all([[1, 2, 3], [0], [0], [0]]))
for g, exp in EXAMPLES:
    p = path_all_nodes(g)
    assert len(p) - 1 == exp and set(p) == set(range(len(g)))
    print("hành trình:", p)

chu trình hình sao: 6
hành trình: [1, 0, 2, 0, 3]
hành trình: [0, 1, 4, 2, 3]
hành trình: [0]
hành trình: [0, 1]
hành trình: [0, 1, 2, 3]


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hoán vị | thứ tự thăm lần đầu + `D` | `O(n!·n)` | `O(n²)` | đối chiếu, `n ≤ 8` |
| 2. BFS `(mask, v)` | đa nguồn, trọng số 1 | `O(2ⁿ·m)` | `O(2ⁿ·n)` | **nộp bài** |
| 3. Held–Karp | `dp[mask][v]` với `D` | `O(2ⁿ·n²)` | `O(2ⁿ·n)` | trọng số tuỳ ý, biến thể chu trình |
| 4. DP cạnh + SPFA | mask tăng, lan trong mask | `O(2ⁿ·n·m)` | `O(2ⁿ·n)` | khi không muốn tính `D` |

## 17. Bài tập

**Bài 1 (đoán output).** Đáp án cho hình sao 6 đỉnh (tâm 0)? Chu trình 5 đỉnh? Đồ thị đầy đủ 4 đỉnh?

**Bài 2 (tính tay).** Với đường thẳng `0-1-2-3-4`, đáp án là gì nếu bắt đầu ở 0? Ở 2? Đáp án đề (bắt đầu tuỳ ý)?

**Bài 3 (điền TODO).** Viết `shortest_from(graph, s)` — bắt buộc bắt đầu ở `s`.

**Bài 4 (tự viết lại).** Viết lại Held–Karp dùng **Floyd–Warshall** để tính `D`.

**Bài 5 (mini challenge).** Đồ thị có trọng số cạnh (`edges = [u, v, w]`), tìm hành trình ngắn nhất thăm mọi đỉnh (bắt đầu tuỳ ý). Dùng Dijkstra trên trạng thái `(mask, v)` và đối chiếu Held–Karp.

In [25]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def shortest_from_todo(graph, s):
    # TODO: BFS một nguồn ((1 << s), s)
    pass

# Bài 4 — TODO
def held_karp_floyd_todo(graph):
    # TODO: D[i][j] = 1 nếu kề; for k, i, j: D[i][j] = min(D[i][j], D[i][k] + D[k][j])
    pass

# Bài 5 — TODO
def weighted_all_nodes_todo(n, edges):
    # TODO: heapq với (dist, mask, v); cạnh (v -> u, w): (mask | 1<<u, u)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: hình sao `k` lá: mỗi lá trừ lá đầu và cuối phải vào và ra.
- Mức 2: `1 + 2·(k−2) + 1 = 2k − 2` với `k = 5` lá.
- Mức 3: `8`, `4`, `3`.

**Bài 2.**
- Mức 1: từ 0: đi thẳng `4` cạnh.
- Mức 2: từ 2: phải đi về một đầu rồi sang đầu kia: `2 + 4 = 6`.
- Mức 3: đáp án đề `4`.

**Bài 3.**
- Mức 1: chỉ khác phần khởi tạo hàng đợi.
- Mức 2: `n == 1` → 0.
- Mức 3: `min_s shortest_from(g, s)` = đáp án đề.

**Bài 4.**
- Mức 1: `D[i][i] = 0`, kề `= 1`, còn lại `inf`.
- Mức 2: ba vòng lặp `k, i, j`.
- Mức 3: phần DP giữ nguyên.

**Bài 5.**
- Mức 1: trạng thái `(mask, v)`, khoảng cách là tổng trọng số.
- Mức 2: Dijkstra; bỏ qua phần tử heap cũ.
- Mức 3: lần đầu lấy ra mask đầy đủ là đáp án.

## 19. Đáp án tham khảo

In [26]:
import heapq

# Bài 1
assert Solution().shortestPathLength(star6) == 8
assert Solution().shortestPathLength(cycle5) == 4
assert Solution().shortestPathLength(complete4) == 3

# Bài 3
def shortest_from(graph, s):
    n = len(graph)
    if n == 1:
        return 0
    full = (1 << n) - 1
    seen = {(1 << s, s)}
    q = deque([(1 << s, s, 0)])
    while q:
        mask, v, d = q.popleft()
        for u in graph[v]:
            nm = mask | (1 << u)
            if nm == full:
                return d + 1
            if (nm, u) not in seen:
                seen.add((nm, u)); q.append((nm, u, d + 1))

# Bài 2
line5 = [[1], [0, 2], [1, 3], [2, 4], [3]]
assert shortest_from(line5, 0) == 4 and shortest_from(line5, 2) == 6 and Solution().shortestPathLength(line5) == 4
for seed in range(50):
    g = random_connected_graph(random.Random(seed).randint(1, 8), 0.3, random.Random(seed))
    assert min(shortest_from(g, s) for s in range(len(g))) == Solution().shortestPathLength(g)

# Bài 4
def held_karp_floyd(graph):
    n = len(graph)
    INF = float('inf')
    D = [[0 if i == j else INF for j in range(n)] for i in range(n)]
    for i in range(n):
        for j in graph[i]:
            D[i][j] = 1
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if D[i][k] + D[k][j] < D[i][j]:
                    D[i][j] = D[i][k] + D[k][j]
    dp = [[INF] * n for _ in range(1 << n)]
    for v in range(n):
        dp[1 << v][v] = 0
    for mask in range(1, 1 << n):
        for v in range(n):
            if dp[mask][v] < INF:
                for u in range(n):
                    if not mask >> u & 1:
                        nm = mask | (1 << u)
                        dp[nm][u] = min(dp[nm][u], dp[mask][v] + D[v][u])
    return min(dp[(1 << n) - 1])

for g, exp in edge:
    assert held_karp_floyd(g) == exp

# Bài 5
def weighted_all_nodes(n, edges):
    adj = [[] for _ in range(n)]
    for u, v, w in edges:
        adj[u].append((v, w)); adj[v].append((u, w))
    full = (1 << n) - 1
    dist = {}
    pq = [(0, 1 << v, v) for v in range(n)]
    for _, m, v in pq:
        dist[(m, v)] = 0
    heapq.heapify(pq)
    while pq:
        d, mask, v = heapq.heappop(pq)
        if mask == full:
            return d
        if d > dist[(mask, v)]:
            continue
        for u, w in adj[v]:
            st = (mask | (1 << u), u)
            if d + w < dist.get(st, float('inf')):
                dist[st] = d + w
                heapq.heappush(pq, (d + w, st[0], u))

def weighted_hk(n, edges):
    INF = float('inf')
    D = [[0 if i == j else INF for j in range(n)] for i in range(n)]
    for u, v, w in edges:
        D[u][v] = min(D[u][v], w); D[v][u] = min(D[v][u], w)
    for k in range(n):
        for i in range(n):
            for j in range(n):
                D[i][j] = min(D[i][j], D[i][k] + D[k][j])
    dp = [[INF] * n for _ in range(1 << n)]
    for v in range(n):
        dp[1 << v][v] = 0
    for mask in range(1, 1 << n):
        for v in range(n):
            if dp[mask][v] < INF:
                for u in range(n):
                    if not mask >> u & 1:
                        dp[mask | 1 << u][u] = min(dp[mask | 1 << u][u], dp[mask][v] + D[v][u])
    return min(dp[(1 << n) - 1])

for seed in range(60):
    r_ = random.Random(seed)
    n = r_.randint(1, 7)
    g = random_connected_graph(n, 0.3, r_)
    es = [[u, v, r_.randint(1, 9)] for u in range(n) for v in g[u] if u < v]
    assert weighted_all_nodes(n, es) == weighted_hk(n, es)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (BFS đa nguồn trên bitmask):**

```python
if n == 1: return 0
full = (1 << n) - 1
seen = [[False]*n for _ in range(1 << n)]
q = deque()
for v in range(n): seen[1 << v][v] = True; q.append((1 << v, v))
steps = 0
while q:
    steps += 1
    for _ in range(len(q)):
        mask, v = q.popleft()
        for u in graph[v]:
            nm = mask | 1 << u
            if nm == full: return steps
            if not seen[nm][u]: seen[nm][u] = True; q.append((nm, u))
```

**Checklist trước khi submit:**

- [ ] Trạng thái là **cặp** `(mask, node)`.
- [ ] BFS **đa nguồn** từ mọi `(1 << v, v)`.
- [ ] `n == 1` → `0`.
- [ ] Held–Karp phải dùng **khoảng cách ngắn nhất** `D`, không chỉ cạnh kề.
- [ ] Dùng mảng `2ⁿ × n` thay vì `set` tuple để nhanh hơn.